# Week 6 studio — REFERENCE SOLUTION (instructor-only)

**Task brief:** [`README.md`](README.md) · **Lesson plan:** [`../../weeks/week-06.md`](../../weeks/week-06.md) · **Given engines:** [`csp.py`](csp.py), [`logic_lm.py`](logic_lm.py)

This notebook is the **teaching walkthrough** of the week-6 CSP / Logic-LM studio. It
**imports the reference code from [`solution.py`](solution.py)** — it never re-pastes it —
so what is shown is byte-for-byte the code the provided test grades. The correctness
guarantee is separate: `python3 studios/_verify_solutions.py week-06` runs the unmodified
`test_csp.py` against `solution.py`.

> Do not distribute. Excluded from students via `studios/.gitignore`.

In [ ]:
# --- bootstrap: put the week folder (for solution/csp/logic_lm) and the repo root
# (for aicourse) on the path, so this runs from anywhere. ---
import sys, pathlib
here = pathlib.Path.cwd()
week = here if (here / "solution.py").exists() else here / "studios" / "week-06"
root = week.parent.parent
for p in (str(week), str(root)):
    if p not in sys.path:
        sys.path.insert(0, p)

import inspect
import csp
from csp import (CSP, backtrack_search, ac3, sudoku_csp, HARD,
                 valid_sudoku, solution_consistent)
import logic_lm
import solution

## Task 1 — model the school-schedule CSP

Two requirements map straight onto the notebook's `CSP`: *all-different* becomes a
complete neighbour graph (the engine's `consistent` check is already all-different), and
*sports before lunch* becomes a unary restriction folded into `Sports`'s domain
(mornings only).

In [ ]:
print(inspect.getsource(solution.scheduling_csp))
c = solution.scheduling_csp()
sol, counter = backtrack_search(c, use_fc=True, use_mrv=True)
for act in solution.ACTIVITIES:
    print(f"  {act:<9} period {sol[act]}")

# mirrors test_scheduling_is_valid_timetable + test_scheduling_sports_before_lunch
assert set(sol) == set(solution.ACTIVITIES)
assert solution_consistent(sol, c)
assert len(set(sol.values())) == len(solution.ACTIVITIES)
assert sol[solution.SPORTS] <= solution.LUNCH_AFTER
print(f"\nok  complete all-different timetable; sports in period {sol[solution.SPORTS]} "
      f"(<= {solution.LUNCH_AFTER}, before lunch)")

### The ablation on our own instance

The scheduling problem is small, so all five configurations solve it — the interesting
ablation is the Sudoku below. Still, the table is what students report for Task 1.

In [ ]:
solution.run_ablation(solution.scheduling_csp)

## Guarantee-fail moment 1 — "complete" says nothing about *when*

Plain backtracking is **complete**: it will solve any solvable puzzle *eventually*. On
the notebook's HARD Sudoku it exhausts a 300k-call budget and never finishes, while
forward-checking + MRV + AC-3 solves it in ~34k. Completeness is not a runtime
promise.

In [ ]:
plain, cp = backtrack_search(sudoku_csp(HARD), limit=300_000)
smart, cs = backtrack_search(sudoku_csp(HARD), use_fc=True, use_mrv=True, use_ac3=True)
assert plain == "TIMEOUT"
assert valid_sudoku(smart, sudoku_csp(HARD))
assert cs["calls"] < cp["calls"]
print(f"plain backtracking : {cp['calls']:,} calls -> TIMEOUT")
print(f"fc + MRV + AC-3    : {cs['calls']:,} calls -> solved")
print("\nGUARANTEE FAIL (on request): a 'complete' method that never finishes in budget.")

## Guarantee-fail moment 2 — LCV, the heuristic that HURTS

LCV ("least constraining value") is *supposed* to speed search up. On HARD it makes
search **strictly worse** — value ordering is a gamble you pay for on every branch. The
notebook measures 34,478 → 56,813 backtrack calls. Report what you measure, not what the
textbook implies. **This is the flagged result of the week.**

In [ ]:
_, no_lcv = backtrack_search(sudoku_csp(HARD), use_fc=True, use_mrv=True)
_, with_lcv = backtrack_search(sudoku_csp(HARD), use_fc=True, use_mrv=True, use_lcv=True)
print(f"fc + MRV        : {no_lcv['calls']:,} calls")
print(f"fc + MRV + LCV  : {with_lcv['calls']:,} calls")
assert with_lcv["calls"] > no_lcv["calls"]
print("\nGUARANTEE FAIL confirmed: LCV made it WORSE. A hoped-for speedup that")
print("did not pay back — the lesson lands only because we measured it.")

### AC-3 shrinks domains before any guess

The inference win the week is named for: arc-consistency prunes the HARD domains before
search even starts a single decision.

In [ ]:
c = sudoku_csp(HARD)
domains = {v: set(c.domains[v]) for v in c.variables}
before = sum(len(x) for x in domains.values())
ok = ac3(c, domains)
after = sum(len(x) for x in domains.values())
assert ok and after < before
print(f"AC-3 shrinks HARD domains {before} -> {after} "
      f"({before-after} values eliminated before any guess)")

## Task 2 — Logic-LM arm B: LLM writes the CSP, our solver owns the guarantee

`solve_arm_b_with_refinement` wires the given `MODEL_PROMPT` + `classify_arm_b` around a
model and adds the **self-refinement loop**: when a reply fails to parse (MALFORMED) or
the solver reports no solution (over-constrained), the concrete error is fed back into the
prompt and the model retries (up to `max_retries`). That feedback loop is Logic-LM's
actual contribution.

In [ ]:
print(inspect.getsource(solution.solve_arm_b_with_refinement))

### The happy path — a *faithful* scripted model (not echo)

To show the pipeline succeed end-to-end without a live model, we feed a tiny scripted
stub that emits the worked example's gold CSP spec as JSON. This is a stand-in for a
model that translated correctly — arm B then returns `OK` on the first attempt, and the
**solver** (not the model) certifies the answer.

In [ ]:
import json

class ScriptedModel:
    '''A stand-in for a faithful translator: returns a fixed CSP-spec JSON.
    NOT a real model — used only to exercise the arm-B happy path headless.'''
    def __init__(self, spec): self._reply = "```json\n" + json.dumps(spec) + "\n```"
    def complete(self, prompt): return self._reply

ex = logic_lm.worked_examples()[0]        # p01, with a known gold answer
faithful = ScriptedModel(ex["spec"])
cat, sol, tries = solution.solve_arm_b_with_refinement(faithful, ex["puzzle"], gold=ex["gold"])
print("puzzle :", ex["puzzle"][:60], "...")
print("result :", cat, "| solution:", sol, "| attempts:", tries)
assert cat == logic_lm.OK and sol == ex["gold"] and tries == 1
print("ok  faithful translation -> OK on attempt 1; the SOLVER certifies the answer")

### The refinement loop under a FAKE model (echo backend)

Now the headless `echo` backend. **`echo` is a FAKE** — it returns prose with no JSON, so
every reply is `MALFORMED`. That is exactly what drives the refinement loop: it retries
`max_retries` times, feeding the parse error back each round, and finally reports the
loud failure. A real model would emit JSON — re-run with `backend="ollama"` or
`backend="manual"` for the genuine arm-B run over `puzzles.json`. The teaching point:
arm B's failures are **loud** (the solver/parser tell you), unlike arm C's silent
wrong answers.

In [ ]:
from aicourse.llm import LLM
echo = LLM(backend="echo")   # FAKE: never a real translation
cat, sol, tries = solution.solve_arm_b_with_refinement(echo, ex["puzzle"], gold=ex["gold"], max_retries=3)
print("echo reply sample:", echo.complete("x").text)
print("\nresult :", cat, "| solution:", sol, "| attempts:", tries)
assert cat == logic_lm.MALFORMED and sol is None and tries == 3
print("ok  fake model never parses -> MALFORMED after retrying all 3 attempts")
print("The failure is LOUD and visible — the whole reason to prefer arm B over arm C.")